<!-- Cache bust 13_context_managers_notebook -->

# Context Managers

***

### 🔹 1. Introduction to Context Managers & the with Statement
When writing robust production programs, **resource management** (opening files, database connections, acquiring thread locks) is critical. If a resource is not closed or released properly, it leaks memory, locks files, and consumes resources.

The manual way to ensure release is using `try-finally`:
```python
file = open("data.txt", "w")
try:
    file.write("hello")
finally:
    file.close() # Always executed
```
Python's **`with`** statement encapsulates this pattern into a clean, readable syntax, acting as a **Context Manager**.

In [ ]:
# Clean resource management with the 'with' statement
with open("data.txt", "w") as file:
    file.write("Hello, Context Managers!")
# The file is automatically closed here!

***

### 🔹 2. Class-Based Context Managers
You can make any class act as a context manager by implementing the **Context Manager Protocol**:
1. **`__enter__(self)`**: Executed when entering the `with` block. Returns the resource (assigned to the variable after `as`).
2. **`__exit__(self, exc_type, exc_value, traceback)`**: Executed when leaving the `with` block.
   - If an exception occurred inside the block, its type, value, and traceback are passed to `__exit__`.
   - If `__exit__` returns `True`, the exception is suppressed. If it returns `False` (or `None`), the exception is raised normally.

In [ ]:
class FileOpener:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode
        
    def __enter__(self):
        print(f"[Enter] Opening file: {self.filename}")
        self.file = open(self.filename, self.mode)
        return self.file
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("[Exit] Closing file...")
        self.file.close()
        if exc_type is not None:
            print(f"[Exit] Exception occurred: {exc_val}")
        return True # Suppress the exception

# Testing our custom context manager
with FileOpener("sample.txt", "w") as f:
    f.write("Custom context manager writing.")
    # Intentionally trigger an exception
    result = 10 / 0

print("Program continues running since exception was suppressed.")

***

### 🔹 3. Generator-Based Context Managers (`contextlib.contextmanager`)
Implementing custom classes with `__enter__` and `__exit__` can be verbose. The `contextlib` module provides a decorator `@contextmanager` that allows you to turn a generator function into a context manager.

#### 🔹 Structure:
1. Everything before the `yield` statement acts as `__enter__`.
2. The value yielded is bound to the `as` variable.
3. Everything after `yield` acts as `__exit__`.
4. Wrapping `yield` in a `try-finally` block is mandatory to guarantee resource release.

In [ ]:
from contextlib import contextmanager

@contextmanager
def db_transaction(connection_name):
    print(f"[DB] Connecting to {connection_name}...")
    db_state = {"connected": True}
    try:
        # Yielding the resource
        yield db_state
    except Exception as e:
        print(f"[DB] Transaction failed: {e}. Rolling back changes.")
    finally:
        print("[DB] Closing connection state...")
        db_state["connected"] = False

# Using generator context manager
with db_transaction("MySQL_Prod") as db:
    print("Is DB active?", db["connected"])
    print("Performing SQL operations...")
    raise RuntimeError("Corrupted query execution")

***

### 🔹 4. Useful contextlib Utilities
The `contextlib` module contains several built-in context managers that solve common programming problems.

#### 🔹 1. `suppress`
Silently ignores specific exceptions without using empty `except: pass` blocks.

In [ ]:
import os
from contextlib import suppress

# Silently ignore FileNotFoundError if the file doesn't exist
with suppress(FileNotFoundError):
    os.remove("non_existent_file.txt")
print("suppress executed cleanly.")

#### 🔹 2. `redirect_stdout`
Redirects standard console output (`print()`) to a file or stream (useful for testing or logging).

In [ ]:
import io
from contextlib import redirect_stdout

f = io.StringIO()
with redirect_stdout(f):
    print("This goes into the buffer, not the console!")
    print("Hello redirected!")

print("Console output resumes. Buffered text:", f.getvalue())

#### 🔹 3. `ExitStack`
Allows you to enter multiple context managers dynamically. It is cleaner than deeply nesting multiple `with` blocks.

In [ ]:
from contextlib import ExitStack

# Open multiple files dynamically without nesting 'with'
files_to_open = ["file1.txt", "file2.txt"]
# Create files
for f in files_to_open:
    with open(f, "w") as fh:
        fh.write(f"Sample data for {f}")

with ExitStack() as stack:
    # Open files dynamically
    opened_files = [stack.enter_context(open(name, "r")) for name in files_to_open]
    for idx, f_obj in enumerate(opened_files):
        print(f"Reading File {idx+1}:", f_obj.read())

***

### 📝 Practice Questions

1. **Execution Timer Context Manager:** Design a class-based context manager `Timer` that measures and prints the elapsed time taken to run code blocks placed inside it.
2. **Directory Changer Context Manager:** Write a generator-based context manager `change_dir(path)` that changes the current working directory to `path` using `os.chdir` upon entering, and automatically returns to the original directory upon exiting.
3. **HTML Tag Wrapper:** Write a generator context manager `html_tag(tag_name)` that prints `<tag_name>` before yield and `</tag_name>` after yield. Test it by nesting tags (e.g., nesting `html_tag('span')` inside `html_tag('p')`).